# Handling Semi Structured Data


In [0]:
from common_utils.scd import scd_type1

In [0]:
df = spark.read.table("retail_data_platform.bronze_layer.cosmosdb_sales_ingestion")

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
"""{"_id": {"$oid": "6a999f4a9e0b277853440b72"}, 

"clicked_items": [["AVpe48Es1cnluZ0-ZHZU", "37"], ["AVpfKEAuilAPnD_xWKpK", "79"], ["AVpe7f2rLJeJML43yi3P", "23"], ["AVqFXQH5nnc1JgDc3gBZ", "63"], ["AVpe48Es1cnluZ0-ZHZU", "39"], ["AVpfMVD-ilAPnD_xW6bu", "53"]],


 "customer_id": 10715991,
 "customer_name": "LYJAK,  KEVIN",
 "number_of_line_items": 3, 
 "order_datetime": 1564760657.0,
 "order_number": 317568079,

 "ordered_products": [{"curr": "USD", "id": "AVqFXQH5nnc1JgDc3gBZ", "name": "Karsair - VENGEANCE LED Series 16GB (2PK 8GB) 3.0GHz DDR4 Desktop Memory with LED Lighting - Black", "price": "366", "promotion_info": null, "qty": "2", "unit": "pcs"}, {"curr": "USD", "id": "AVpe48Es1cnluZ0-ZHZU", "name": "Zamaha RX-V479BL 5.1-Channel AV Receiver (Black)", "price": "27", "promotion_info": null, "qty": "2", "unit": "pcs"}, {"curr": "USD", "id": "AVpfMVD-ilAPnD_xW6bu", "name": "Rony - BC-TRX Battery Charger - Black", "price": "181", "promotion_info": {"promo_disc": 0.03, "promo_id": "0", "promo_item": "AVpfMVD-ilAPnD_xW6bu", "promo_qty": "3"}, "qty": "3", "unit": "pcs"}], 
 
 "promo_info": [{"promo_disc": 0.03, "promo_id": "0", "promo_item": "AVpfMVD-ilAPnD_xW6bu", "promo_qty": "3"}]}"""

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    LongType,
    DoubleType,
    IntegerType,
    ArrayType
)

# Schema for promotion information
promotion_schema = StructType([
    StructField("promo_disc", DoubleType(), True),
    StructField("promo_id", StringType(), True),
    StructField("promo_item", StringType(), True),
    StructField("promo_qty", StringType(), True)
])

# Main JSON schema
json_schema = StructType([

    # _id → object → $oid
    StructField(
        "_id",
        StructType([
            StructField("$oid", StringType(), True)
        ]),
        True
    ),

    # Array of arrays of strings
    # Example: [["product_id", "37"], ["product_id", "79"]]
    StructField(
        "clicked_items",
        ArrayType(
            ArrayType(StringType())
        ),
        True
    ),

    # Customer information
    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("number_of_line_items", IntegerType(), True),

    # Decimal number
    StructField("order_datetime", DoubleType(), True),

    # Large whole number
    StructField("order_number", LongType(), True),

    # Array of ordered product objects
    StructField(
        "ordered_products",
        ArrayType(
            StructType([
                StructField("curr", StringType(), True),
                StructField("id", StringType(), True),
                StructField("name", StringType(), True),
                StructField("price", StringType(), True),

                # promotion_info is an object
                StructField(
                    "promotion_info",
                    promotion_schema,
                    True
                ),

                StructField("qty", StringType(), True),
                StructField("unit", StringType(), True)
            ])
        ),
        True
    ),

    # Array of promotion objects
    StructField(
        "promo_info",
        ArrayType(promotion_schema),
        True
    )
])

In [0]:
import pyspark.sql.functions as f
df_parsed = df.withColumn("data", f.from_json(df["json_data"],json_schema))

In [0]:
df_parsed.printSchema()


In [0]:
df_order_products = df_parsed.select(
    f.col("data.customer_id").alias("customer_id"),
    f.col("data.customer_name").alias("customer_name"),
    f.col("data.order_number").alias("order_number"),
    f.explode(f.col("data.ordered_products")).alias("products")
    
)

In [0]:
df_order_products = df_order_products.select(
    f.col("customer_id").alias("customer_id"),
    f.col("customer_name").alias("customer_name"),
    f.col("order_number").alias("order_number"),
    f.col("products.curr").alias("currency"),
    f.col("products.id").alias("product_id"),
    f.col("products.name").alias("product_name"),
    f.col("products.price").alias("price"),
    f.col("products.qty").alias("quantity"),
    f.col("products.unit").alias("unit"),

)


In [0]:
def transform_customer_name(df):
    from pyspark.sql import functions as f

    return df.withColumn(
        "customer_name",
        f.when(
            f.col("customer_name").contains(","),
            f.concat(
                f.trim(f.element_at(f.split(f.lower("customer_name"), ","), 2)),
                f.lit(" "),
                f.trim(f.element_at(f.split(f.lower("customer_name"), ","), 1))
            )
        ).otherwise(
            f.lower(f.col("customer_name"))
        )
    )

In [0]:
cleaned_df =transform_customer_name(df_order_products)
cleaned_df.display()

In [0]:
target_table ="retail_data_platform.silver.orders_products"
condition = "target.customer_id = source.customer_id"
source = transform_customer_name(df_order_products)

from common_utils.scd import scd_type1
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")


scd_type1(spark,source,target_table,condition)

| # | Column                 | Source / Transformation                            |
| - | ---------------------- | -------------------------------------------------- |
| 1 | `order_number`         | Bronze `order_number`                              |
| 2 | `customer_id`          | Bronze `customer_id`                               |
| 3 | `customer_name`        | Bronze `customer_name`                             |
| 4 | `number_of_line_items` | Bronze `number_of_line_items`                      |
| 5 | `order_timestamp`      | `order_datetime` converted using `from_unixtime()` |


In [0]:
df_sales_orders = df_parsed.select(
    f.col("data.order_number").alias("order_number"),
    f.col("data.customer_id").alias("customer_id"),
    f.col("data.customer_name").alias("customer_name"),
    f.col("data.number_of_line_items").alias("number_of_line_items"),
    f.col("data.order_datetime").alias("order_timestamp")
)
df_sales_orders.display()
"""
1-customer name should be small
2-correct the order of the name
"""

In [0]:

target_table ="retail_data_platform.silver.sales_orders"
condition = "target.customer_id = source.customer_id"
source = transform_customer_name(df_sales_orders)

from common_utils.scd import scd_type1
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")


scd_type1(spark,source,target_table,condition)

In [0]:
df_promotions = df_parsed.select(
    f.col("data.customer_id").alias("customer_id"),
    f.col("data.order_number").alias("order_number"),
    f.explode(f.col("data.promo_info")).alias("promo")
)



| # | Column             | Source / Transformation              |
| - | ------------------ | ------------------------------------ |
| 1 | `customer_id`      | Bronze `customer_id`                 |
| 2 | `order_number`     | Bronze `order_number`                |
| 3 | `promo_id`         | `promo_info.promo_id`                |
| 4 | `promo_product_id` | `promo_info.promo_item`              |
| 5 | `discount`         | `promo_info.promo_disc`              |
| 6 | `promo_quantity`   | `promo_info.promo_qty` cast to `int` |


In [0]:
df_promotions = df_promotions.select(
    f.col("customer_id").alias("customer_id"),
    f.col("order_number").alias("order_number"),
    f.col("promo.promo_id").alias("promo_id"),
    f.col("promo.promo_item").alias("promo_product_id"),
    f.col("promo.promo_disc").alias("discount"),
    f.col("promo.promo_qty").alias("promo_quantity")
)



In [0]:
df_promotions.printSchema()
df_promotions.display()
"""
mayby drop is the order number is duplicated
"""
target_table ="retail_data_platform.silver.promotions_info"
condition = "target.customer_id = source.customer_id"
source = transform_customer_name(df_promotions)

from common_utils.scd import scd_type1
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")


| # | Column          | Source / Transformation |
| - | --------------- | ----------------------- |
| 1 | `customer_id`   | Bronze `customer_id`    |
| 2 | `customer_name` | Bronze `customer_name`  |
| 3 | `order_number`  | Bronze `order_number`   |
| 4 | `product_id`    | `clicked_items[0]`      |
| 5 | `score`         | `clicked_items[1]`      |


In [0]:
df_clicked_items = df_parsed.select(
    f.col("data.customer_id").alias("customer_id"),
    f.col("data.customer_name").alias("customer_name"),
    f.col("data.order_number").alias("order_number"),
    f.explode(f.col("data.clicked_items")).alias("clicked_items")

)


In [0]:
df_clicked_items = df_clicked_items.select(
    f.col("customer_id"),
     f.col("customer_name"),
    f.col("order_number"),
    f.col("clicked_items")[0].alias("product_id"),
    f.col("clicked_items")[1].alias("score")
)


In [0]:
"""
1-customer name should be small
2-correct the order of the name
"""
df_clicked_items = transform_customer_name(df_clicked_items)
df_clicked_items.display()


In [0]:
target_table ="retail_data_platform.silver.sales_orders"
condition = "target.customer_id = source.customer_id"
source = transform_customer_name(df_clicked_items)

from common_utils.scd import scd_type1
df = spark.read.table("retail_data_platform.bronze_layer.sqlserver_customers")
